# Logistic Regression

## Objective

The objective of this notebook is to implement a multiclass Logistic Regression model from scratch using NumPy. The implementation includes the Softmax activation function, Cross-Entropy Loss, Gradient Descent optimization, and prediction without using any machine learning libraries.

In [1]:
# Import Required Libraries

import numpy as np
import pandas as pd
import pickle

In [2]:
# Load scaled datasets

X_train = pd.read_csv("../data/processed/X_train_scaled.csv")
X_test = pd.read_csv("../data/processed/X_test_scaled.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("Training Data :", X_train.shape)
print("Testing Data  :", X_test.shape)

Training Data : (1760, 7)
Testing Data  : (440, 7)


In [3]:
# Convert DataFrames into NumPy arrays

X_train = X_train.to_numpy()
X_test = X_test.to_numpy()

y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

## One-Hot Encoding

The target labels are converted into one-hot encoded vectors for multiclass classification.

In [4]:
def one_hot_encode(y, num_classes):

    encoded = np.zeros((len(y), num_classes))

    for i in range(len(y)):
        encoded[i][y[i]] = 1

    return encoded

In [5]:
num_classes = len(np.unique(y_train))

y_train_encoded = one_hot_encode(y_train, num_classes)

## Softmax Function

Softmax converts raw model outputs into probability values that sum to 1 across all classes.

In [6]:
def softmax(z):

    z = z - np.max(z, axis=1, keepdims=True)

    exp = np.exp(z)

    return exp / np.sum(exp, axis=1, keepdims=True)

## Initialize Parameters

Initialize the model weights and bias with zeros.

In [7]:
n_features = X_train.shape[1]
n_classes = num_classes

weights = np.zeros((n_features, n_classes))
bias = np.zeros((1, n_classes))

## Cross-Entropy Loss

Cross-Entropy Loss measures the difference between predicted probabilities and the true class labels.

In [8]:
def cross_entropy_loss(y_true, y_pred):

    epsilon = 1e-15

    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    loss = -np.mean(
        np.sum(
            y_true * np.log(y_pred),
            axis=1
        )
    )

    return loss

## Gradient Computation

Compute the gradients of the loss function with respect to the model parameters.

In [9]:
def compute_gradients(X, y_true, y_pred):

    m = X.shape[0]

    dw = (1 / m) * np.dot(X.T, (y_pred - y_true))

    db = (1 / m) * np.sum(
        y_pred - y_true,
        axis=0,
        keepdims=True
    )

    return dw, db

## Training Function

Train the Logistic Regression model using Gradient Descent.

In [10]:
#Training
def train(
    X,
    y,
    learning_rate=0.01,
    epochs=1000
):

    global weights
    global bias

    losses = []

    for epoch in range(epochs):

        logits = np.dot(X, weights) + bias

        predictions = softmax(logits)

        loss = cross_entropy_loss(
            y,
            predictions
        )

        dw, db = compute_gradients(
            X,
            y,
            predictions
        )

        weights -= learning_rate * dw
        bias -= learning_rate * db

        losses.append(loss)

        if epoch % 100 == 0:

            print(
                f"Epoch {epoch} | Loss = {loss:.4f}"
            )

    return losses

In [11]:
#Train Model
losses = train(
    X_train,
    y_train_encoded,
    learning_rate=0.05,
    epochs=1000
)

Epoch 0 | Loss = 3.0910
Epoch 100 | Loss = 3.0279
Epoch 200 | Loss = 2.9695
Epoch 300 | Loss = 2.9144
Epoch 400 | Loss = 2.8618
Epoch 500 | Loss = 2.8116
Epoch 600 | Loss = 2.7636
Epoch 700 | Loss = 2.7177
Epoch 800 | Loss = 2.6738
Epoch 900 | Loss = 2.6317


In [12]:
#prediction
def predict(X):

    logits = np.dot(X, weights) + bias

    probabilities = softmax(logits)

    return np.argmax(
        probabilities,
        axis=1
    )

In [13]:
#Accuracy
def accuracy(y_true, y_pred):

    return np.mean(y_true == y_pred)

In [14]:
#Evaluate
train_prediction = predict(X_train)
test_prediction = predict(X_test)

print(
    "Training Accuracy:",
    accuracy(
        y_train,
        train_prediction
    )
)

print(
    "Testing Accuracy:",
    accuracy(
        y_test,
        test_prediction
    )
)

Training Accuracy: 0.647159090909091
Testing Accuracy: 0.6022727272727273


In [15]:
#save model
model = {
    "weights": weights,
    "bias": bias
}

with open(
    "../saved_models/logistic_regression.pkl",
    "wb"
) as file:

    pickle.dump(model, file)

print("Model Saved Successfully.")

Model Saved Successfully.
